In [1]:
!pip install -q ddgs ollama
!apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q statsmodels


!pkill -f "ollama serve"
import time
time.sleep(2)

import subprocess, time

with open('/kaggle/working/ollama.log', 'w') as log_file:
    subprocess.Popen(['ollama', 'serve'], stdout=log_file, stderr=log_file)

time.sleep(5)
print("Service Ollama demarre")

# Gemma 3
!ollama pull gemma3:1b
!ollama pull gemma3:4b
!ollama pull gemma3:12b

# Qwen 2.5
!ollama pull qwen2.5:1.5b
!ollama pull qwen2.5:7b
!ollama pull qwen2.5:14b

# Llama 3.2
!ollama pull llama3.2:1b
!ollama pull llama3.2:3b

# Phi-4
!ollama pull phi4-mini


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 94.6 MB/s eta 0:00:00:00:0100:01
Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Service Ollama demarre
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manife

In [2]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import csv
import random
import re
import time
import ollama
from ddgs import DDGS
from collections import Counter
from statsmodels.stats.contingency_tables import cochrans_q
from scipy.stats import chi2
import numpy as np
import os, datetime
from urllib.parse import urlparse

In [ ]:
!nvidia-smi

In [4]:
def lire_csv(chemin):
    with open(chemin, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

In [5]:
chemin_in = "/kaggle/input/datasets/imenec/data-cat/"
chemin_out = "/kaggle/working/"

articles_B = lire_csv(chemin_in + "cat_B_150.csv")
articles_D = lire_csv(chemin_in + "cat_D_150.csv")

print(f"Catégorie B : {len(articles_B)} articles")
print(f"Catégorie D : {len(articles_D)} articles")

Catégorie B : 150 articles
Catégorie D : 150 articles


In [ ]:
def chercher_preuves(requete):
    # cherche sur web des infos sur le texte passé en param
    # retourne les 3 premiers resultats trouvés formatés en liste de points

    #requete = titre if titre else texte[:100]
    try:
        resultats = DDGS().text(requete, max_results=3)
    except Exception as e:
        print(f"  Erreur de recherche : {e}")
        return "(aucune preuve trouvee)"

    if not resultats:
        return "(aucune preuve trouvee)"

    morceaux = []
    for r in resultats:
        morceaux.append(f"- {r.get('title', '')} : {r.get('body', '')[:200]}")
    return "\n".join(morceaux)


In [7]:
def chercher_preuves_2(requete, max_tentatives=3):
    # cherche sur web des infos sur le texte passé en param
    # retourne les 3 premiers resultats trouvés formatés en liste de points
    # difference avec l'autre, si on échoue a trouver des preuves, on essaye jusqu'a 3 tentatives


    for tentative in range(1, max_tentatives + 1):
        try:
            # timeout augmente a 20s (au lieu de 5s par defaut)
            with DDGS(timeout=20) as ddgs:
                resultats = ddgs.text(query=requete, max_results=3)

            if not resultats:
                return "(aucune preuve trouvee)", []

            morceaux = []
            domaines = []
            
            for r in resultats:
                morceaux.append(f"- {r.get('title', '')} : {r.get('body', '')[:200]}")
                domaines.append(urlparse(r.get("href", "")).netloc)
            return "\n".join(morceaux), domaines

        except Exception as e:
            print(f"  [!] Tentative {tentative}/{max_tentatives} echouee : {e}")
            if tentative < max_tentatives:
                # attend de plus en plus longtemps a chaque tentative (3s, 6s, 9s)
                time.sleep(3 * tentative)

    # si toutes les tentatives ont echoue
    return "(aucune preuve trouvee apres plusieurs tentatives)", []


In [8]:
def construire_prompt(titre, texte, source, preuves, avec_rag):
    texte_court = texte[:800]

    prompt = "Voici un article a verifier.\n\n"
    prompt = prompt + "Source : " + source + "\n"
    prompt = prompt + "Titre : " + titre + "\n"
    prompt = prompt + "Texte : " + texte_court + "\n\n"

    if avec_rag:
        prompt = prompt + "Preuves trouvees sur le web concernant cette affirmation :\n"
        prompt = prompt + preuves + "\n\n"
        prompt = prompt + "En te basant sur ces preuves, verifie si l'affirmation de cet article est vraie ou fausse.\n"
    else:
        prompt = prompt + "En te basant sur ce que tu connais déja, verifie si l'affirmation de cet article est vraie ou fausse.\n"
    
    prompt = prompt + "Reponds uniquement par un seul mot : \"fake\" ou \"real\"."

    return prompt


def extraire_verdict(reponse_llm):
    texte = reponse_llm.lower()
    if re.search(r"\bfake\b", texte):
        return "fake"
    if re.search(r"\breal\b", texte):
        return "real"
    return "indetermine"


def extraire_verdict_2(reponse_llm):
    texte = reponse_llm.lower().strip()
    # Si la reponse est longue, c'est surement pas un real ou un fake 
    if len(texte.split()) > 5:
        return "indetermine"
    if re.search(r"\bfake\b", texte):
        return "fake"
    if re.search(r"\breal\b", texte):
        return "real"
    return "indetermine"

In [9]:
def demander_verdict(titre, texte, source, preuves, avec_rag):

    prompt = construire_prompt(titre, texte, source, preuves, avec_rag)

    reponse = ollama.chat(model=m, messages=[{"role": "user", "content": prompt}])

    contenu = reponse["message"]["content"]

    return extraire_verdict(contenu)


In [10]:
def verdict_n_fois(titre, texte, source, preuves, avec_rag, n=5):
    # Fonction qui appelle le LLM n fois avec les memes parametres
    # et renvoie le verdict 
    # on lance n fois pour chercher une certaine stabilité dans les reponses du modele
    # apres je prends la verdict majoritaire retourné

    verdicts = []
    for i in range(n):
        v = demander_verdict(titre, texte, source, preuves, avec_rag)
        verdicts.append(v)
        
    # compte combien a chaque fois cahque valeur apparait dans la liste
    compte = Counter(verdicts)
    # retourne l'element le plus frequent, [0][0] pour chercher l'etiquette real ou fake
    majorite = compte.most_common(1)[0][0]
    return majorite, verdicts

In [21]:
FICHIER_PREUVES = chemin_out + "preuves_D.csv"

def sauver_preuve(id_article, requete, preuves, domaines):
    nouveau = not os.path.exists(FICHIER_PREUVES)
    with open(FICHIER_PREUVES, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nouveau:
            w.writerow(["id", "requete", "Preuves", "domaine", "date_recherche"])
        w.writerow([id_article, requete, preuves, " | ".join(domaines), datetime.date.today().isoformat()])

In [16]:
def nettoyer_requete(titre):
    # enlève les noms de sites de fact-checking qui revelent la source
    titre = re.sub(r"gossip\s*cop|politi\s*fact|snopes", "", titre, flags=re.IGNORECASE)
    titre = re.sub(r"\s+", " ", titre)      #espaces en double
    return titre.strip(" -|:–—")              # tirets ou deux-points restes aux bords

In [17]:
def tester_contrefactuel(article, source_2, avec_rag, creer_preuves, pas_source="source non identifiée"):
    
    # exemple de source peu connue : yournewswire.com
    # exemple de source connue : dailymail.co.uk
    # Je teste un article 3 fois : 
    # une fois avec sa vraie source originale,
    # une fois avec une source oposée au statut de la source originale
    # si la source originale est connue, on changera pour une source peu_connue et virce versa
    # une fois en anonymisant la source en mettant a la place "source non identifiée"
    # mais en gardant le meme contenu, meme preuves.
    
    titre = article.get("title", "")
    texte = article.get("text", "")
    vraie_source = article.get("domaine", "source inconnue")
    vrai_label = article.get("label", "")

    # RAG ou non-RAG
    if avec_rag:
        if creer_preuves:
            # la 1ere config de Gemma cherche les preuves une seule fois
            #requete = titre if titre else texte[:100]
            requete = nettoyer_requete(titre) if titre else texte[:100]
            preuves, domaines = chercher_preuves_2(requete)
            
            # on sauvegarde les preuves en memoire dans l'article
            article["preuves"] = preuves

            # on sauvegarde les preuves avec leurs détails dans le fichiers csv
            sauver_preuve(article["id"], requete, preuves, domaines)
            print(f"  RAG CRÉÉ pour : {article.get('id', '')}")
        else:
            # les autres modele vont reutiliser les preuves de Gemma
            preuves = article["preuves"]
            print(f"  RAG RÉUTILISÉ pour : {article.get('id', '')}")
    else:
        preuves = None


    # Version 1 : avec la vraie source 
    verdict_original, n_original = verdict_n_fois(titre, texte, vraie_source, preuves, avec_rag)

    # Version 2 : source changée pour l'opposé de la source originale
    verdict_oppose, n_oppose = verdict_n_fois(titre, texte, source_2, preuves, avec_rag)

    # Version 3 : source rendue anonyme
    verdict_anonyme, n_anonyme = verdict_n_fois(titre, texte, pas_source, preuves, avec_rag)
    

    # flip = oui sil y a au moins un verdict different entre les 3 rendus
    if verdict_original == verdict_oppose == verdict_anonyme:
        flip = "non"
    else:
        flip = "oui"

    return {
        "id": article.get("id", ""),
        "vraie_source": vraie_source,
        "vrai_label": vrai_label,
        "verdict_original": verdict_original,
        "verdict_oppose": verdict_oppose,
        "verdict_anonyme": verdict_anonyme,
        "flip": flip,
    }

In [18]:
def lancer_test_contrefactuel(echantillon, chemin_sortie, source_2, avec_rag, creer_preuves):

    colonnes = ["id", "vraie_source", "vrai_label", "verdict_original", "verdict_oppose" ,"verdict_anonyme", "flip"]
    resultats = []

    with open(chemin_sortie, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=colonnes)
        writer.writeheader()

        for i, article in enumerate(echantillon):
            r = tester_contrefactuel(article, source_2, avec_rag, creer_preuves)
            writer.writerow(r)
            resultats.append(r)
            print(f"[{i+1}/{len(echantillon)}] {r['vraie_source']} -> vraiLabel={r['vrai_label']}, original={r['verdict_original']}, oppose={r['verdict_oppose']} ,anonyme={r['verdict_anonyme']}, flip={r['flip']}")
            
    return resultats

In [19]:

MODELES = ["gemma3:4b", "gemma3:1b", "gemma3:12b",
           "qwen2.5:1.5b", "qwen2.5:7b", "qwen2.5:14b",
           "llama3.2:1b", "llama3.2:3b", "phi4-mini"]

preuve = True  #True seulement pour le 1er appel RAG

for m in MODELES:
    nom = m.replace(":", "_")

    print("==================== AVEC RAG :", m,"=================================")
    
    lancer_test_contrefactuel(echantillon=articles_B,
                              chemin_sortie=chemin_out + nom + "_B_RAG.csv",
                              source_2="yournewswire.com",
                              avec_rag=True, 
                              creer_preuves=preuve)
    
    preuve = False

    print("==================== SANS RAG :", m,"=================================")
    
    lancer_test_contrefactuel(echantillon=articles_B,
                              chemin_sortie=chemin_out + nom + "_B_WRAG.csv",
                              source_2="yournewswire.com",
                              avec_rag=False, 
                              creer_preuves=preuve)

==================== AVEC RAG : gemma3:4b =================================
  RAG CRÉÉ pour : gossipcop-898083
[1/150] cosmopolitan.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
  RAG CRÉÉ pour : gossipcop-931703
[2/150] dailymail.co.uk -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
  [!] Tentative 1/3 echouee : No results found.
  RAG CRÉÉ pour : gossipcop-9940338306
[3/150] ibtimes.co.uk -> vraiLabel=fake, original=real, oppose=real ,anonyme=real, flip=non
  RAG CRÉÉ pour : gossipcop-948850
[4/150] elle.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
  [!] Tentative 1/3 echouee : No results found.
  [!] Tentative 2/3 echouee : No results found.
  RAG CRÉÉ pour : gossipcop-908965
[5/150] dailymail.co.uk -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
  RAG CRÉÉ pour : gossipcop-4170688503
[6/150] dailymail.co.uk -> vraiLabel=fake, original=real, oppose=real ,anonyme=real, flip=non
  RAG 

h2 connection driver error: connection reset


  [!] Tentative 1/3 echouee : No results found.
  [!] Tentative 2/3 echouee : error sending request for url (https://html.duckduckgo.com/html/) > operation timed out
  RAG CRÉÉ pour : gossipcop-3178329969
[80/150] eonline.com -> vraiLabel=fake, original=real, oppose=real ,anonyme=real, flip=non
  RAG CRÉÉ pour : gossipcop-874258
[81/150] elle.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
  [!] Tentative 1/3 echouee : No results found.
  RAG CRÉÉ pour : gossipcop-941490
[82/150] dailymail.co.uk -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
  [!] Tentative 1/3 echouee : error sending request for url (https://html.duckduckgo.com/html/) > operation timed out
  [!] Tentative 2/3 echouee : error sending request for url (https://html.duckduckgo.com/html/) > operation timed out
  [!] Tentative 3/3 echouee : error sending request for url (https://html.duckduckgo.com/html/) > operation timed out
  RAG CRÉÉ pour : gossipcop-6813792423
[83/150] c

In [22]:
MODELES = ["gemma3:1b", "gemma3:4b", "gemma3:12b",
           "qwen2.5:1.5b", "qwen2.5:7b", "qwen2.5:14b",
           "llama3.2:1b", "llama3.2:3b", "phi4-mini"]

preuve = True  #True seulement pour le 1er appel RAG

for m in MODELES:
    nom = m.replace(":", "_")

    print("==================== AVEC RAG :", m,"=================================")
    
    lancer_test_contrefactuel(echantillon=articles_D,
                              chemin_sortie=chemin_out + nom + "_D_RAG.csv",
                              source_2="dailymail.co.uk",
                              avec_rag=True, 
                              creer_preuves=preuve)
    
    preuve = False

    print("==================== SANS RAG :", m,"=================================")
    
    lancer_test_contrefactuel(echantillon=articles_D,
                              chemin_sortie=chemin_out + nom + "_D_WRAG.csv",
                              source_2="dailymail.co.uk",
                              avec_rag=False, 
                              creer_preuves=preuve)

==================== AVEC RAG : gemma3:1b =================================


h2 connection driver error: connection reset


  [!] Tentative 1/3 echouee : No results found.
  [!] Tentative 2/3 echouee : error sending request for url (https://html.duckduckgo.com/html/) > operation timed out
  [!] Tentative 3/3 echouee : error sending request for url (https://html.duckduckgo.com/html/) > operation timed out
  RAG CRÉÉ pour : gossipcop-938483
[1/150] lastylemagazine.com -> vraiLabel=real, original=fake, oppose=fake ,anonyme=fake, flip=non
  [!] Tentative 1/3 echouee : error sending request for url (https://html.duckduckgo.com/html/) > operation timed out
  [!] Tentative 2/3 echouee : error sending request for url (https://html.duckduckgo.com/html/) > operation timed out
  [!] Tentative 3/3 echouee : error sending request for url (https://html.duckduckgo.com/html/) > operation timed out
  RAG CRÉÉ pour : gossipcop-875218
[2/150] celebritynetworth.com -> vraiLabel=real, original=fake, oppose=fake ,anonyme=fake, flip=non
  RAG CRÉÉ pour : gossipcop-905464
[3/150] romper.com -> vraiLabel=real, original=fake, oppose

In [ ]:
!ls -lh /kaggle/working/

**TEST COCHRAN**


In [ ]:
def calcul_cochran_q(chemin_csv, nom, col_connu, col_peu_connu, col_anonyme):
    articles = lire_csv(chemin_csv)

    tab=[]
    for a in articles:
        if a[col_connu] == "real":
            connu = 1
        else:
            connu = 0

        if a[col_peu_connu] == "real":
            peu_connu = 1
        else:
            peu_connu = 0

        if a[col_anonyme] == "real":
            anonyme = 1
        else:
            anonyme = 0

        tab.append([connu, peu_connu, anonyme])
        
    resultat = cochrans_q(tab)

    print(f"=== {nom} ===")
    print(f"  Statistique Q : {resultat.statistic:.2f}")
    print(f"  p-value : {resultat.pvalue:.2e}")

    if resultat.pvalue < 0.001:
        print("  => Les 3 conditions different significativement entre elles (p<0.001)")
    else:
        print("  => Pas de difference significative globale")
    print()

    return resultat
    

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/"

resultat_B_avecRAG = calcul_cochran_q(
    chemin_csv=chemin_in + "qwen_B_RAG.csv",
    nom="Categorie B (avec RAG)",
    col_connu="verdict_original",
    col_peu_connu="verdict_oppose",
    col_anonyme="verdict_anonyme"
)

resultat_D_avecRAG = calcul_cochran_q(
    chemin_csv=chemin_in + "qwen_D_RAG.csv",
    nom="Categorie D (avec RAG)",
    col_connu="verdict_oppose",
    col_peu_connu="verdict_original",
    col_anonyme="verdict_anonyme"
)

resultat_B_sansRAG = calcul_cochran_q(
    chemin_csv=chemin_in + "qwen_B_WRAG.csv",
    nom="Categorie B (sans RAG)",
    col_connu="verdict_original",
    col_peu_connu="verdict_oppose",
    col_anonyme="verdict_anonyme"
)

resultat_D_sansRAG = calcul_cochran_q(
    chemin_csv=chemin_in + "qwen_D_WRAG.csv",
    nom="Categorie D (sans RAG)",
    col_connu="verdict_oppose",
    col_peu_connu="verdict_original",
    col_anonyme="verdict_anonyme"
)

**Test mcneamar**

In [ ]:
 
def comparer_deux_colonnes(chemin_csv, nom_colonne_1, nom_colonne_2):
    # cette fonction compare 2 colonnes dun fichier csv avec le test mcNeamar
    # la fonction retourne un dictionnaire avec b, c, p-value,
    # proportions "real" de chaque colonne, diff + IC95%
    # b: nb d'articles "real" (colonne 1) devenus "fake" (colonne 2)
    # c: nb d'articles "fake" (colonne 1) devenus "real" (colonne 2)
    # p_value: probabilite d'observer ce desequilibre entre b et c
    # p_real_colonne_1: proportion d'articles "real" dans la colonne 1
    # p_real_colonne_2: proportion d'articles "real" dans la colonne 2
    # diff_proportions: variation de la proportion de "real" (colonne_2 - colonne_1)
    # ic95_low/high:(95% de confiance) la plus petite et la plus grande valeur probable de la "vraie" difference
 
    col1, col2 = [], []
    with open(chemin_csv, newline="", encoding="utf-8") as f:
        for ligne in csv.DictReader(f):
            col1.append(ligne[nom_colonne_1].strip().lower())
            col2.append(ligne[nom_colonne_2].strip().lower())
 
    n = len(col1)
 
    table = {"fake_fake": 0, "fake_real": 0, "real_fake": 0, "real_real": 0}
    for a, b_ in zip(col1, col2):
        table[f"{a}_{b_}"] += 1
 
    b = table["real_fake"]   # col1=real, col2=fake 
    c = table["fake_real"]   # col1=fake, col2=real
    nbr_flip = b + c
 
    # mcnemar
    if nbr_flip == 0:
        stat, p = 0.0, 1.0
    else:
        stat = (abs(b - c) - 1) ** 2 / nbr_flip
        p = 1 - chi2.cdf(stat, df=1)
 
    # proportions + IC95% de la difference appariee
    p1 = col1.count("real") / n #proportion d'articles "real" dans la colonne 1
    p2 = col2.count("real") / n #proportion d'articles "real" dans la colonne 2
    
    diff = p2 - p1
    
    se = ((b + c) - (b - c) ** 2 / n) ** 0.5 / n
    ic_low, ic_high = diff - 1.96 * se, diff + 1.96 * se
 
    return {
        "colonne_1": nom_colonne_1,
        "colonne_2": nom_colonne_2,
        "b_real_to_fake": b,
        "c_fake_to_real": c,
        "p_value": p,
        "p_real_colonne_1": p1,
        "p_real_colonne_2": p2,
        "diff_proportions": diff,
        "ic95_low": ic_low,
        "ic95_high": ic_high,
    }

In [ ]:
 
def afficher_resultat(res, alpha=0.05/3):
    #Afficher les resultats de comparer_deux_colonnes() de facon lisible
    
    print(f"Comparaison : {res['colonne_1']} vs {res['colonne_2']}")
    print(f"  b (real->fake) = {res['b_real_to_fake']}, c (fake->real) = {res['c_fake_to_real']}, "
          f"changements = {res['b_real_to_fake'] + res['c_fake_to_real']}")
    print(f"  McNemar : p = {res['p_value']:.4g} "
          f"-> {'significatif' if res['p_value'] < alpha else 'non significatif'} (alpha={alpha})")
    print(f"  Proportion 'real' : {res['p_real_colonne_1']*100:.1f}% -> {res['p_real_colonne_2']*100:.1f}%")
    print(f"  Difference = {res['diff_proportions']*100:+.1f} points, "
          f"IC95% = [{res['ic95_low']*100:+.1f} ; {res['ic95_high']*100:+.1f}]")
    print()
 

 

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/gemma_B_RAG.csv"
print("Gemma sur catégorie B avec RAG")
# cat B , original = connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_oppose")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/gemma_B_WRAG.csv"
print("Gemma sur catégorie B sans RAG")
# cat B , original = connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_oppose")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/gemma_D_RAG.csv"
print("Gemma sur catégorie D avec RAG")
# cat D , original = peu connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_original")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/gemma_D_WRAG.csv"
print("Gemma sur catégorie D sans RAG")
# cat D , original = peu connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_original")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/qwen_B_RAG.csv"
print("Qwen sur catégorie B avec RAG")
# cat B , original = connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_oppose")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/qwen_B_WRAG.csv"
print("Qwen sur catégorie B sans RAG")
# cat B , original = connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_oppose")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/qwen_D_RAG.csv"
print("Qwen sur catégorie D avec RAG")
# cat D , original = peu connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_original")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/qwen_D_WRAG.csv"
print("Qwen sur catégorie D sans RAG")
# cat D , original = peu connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_original")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res3)